# Train Custom Bird Detector (YOLOv8n)

Fine-tune YOLOv8n on 283 manually reviewed feeder images to replace the generic COCO bird detector.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
!pip install ultralytics -q

In [ ]:
# Upload dataset.zip (85MB)
from google.colab import files
uploaded = files.upload()

In [ ]:
!unzip -q dataset.zip
!echo "Train images:" && ls dataset/images/train/ | wc -l
!echo "Val images:" && ls dataset/images/val/ | wc -l
!cat dataset/dataset.yaml

In [ ]:
# Fix dataset.yaml path for Colab
import yaml
with open('dataset/dataset.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['path'] = '/content/dataset'
with open('dataset/dataset.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('Updated path to /content/dataset')
!cat dataset/dataset.yaml

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # pretrained nano
results = model.train(
    data='dataset/dataset.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    patience=30,        # early stop if val loss plateaus
    freeze=10,          # freeze backbone — critical for small dataset
    close_mosaic=10,    # disable mosaic last 10 epochs
    augment=True,       # mosaic, flip, scale, color jitter
    device=0,           # GPU
    project='bird_detector',
    name='feeder_v1'
)

In [ ]:
# Check results
print('\n=== Results ===')
for k, v in results.results_dict.items():
    if 'map' in k.lower() or 'precision' in k.lower() or 'recall' in k.lower():
        print(f'  {k}: {v:.4f}')
print('\nmAP@50 > 0.80 = good, > 0.90 = excellent')

In [ ]:
# Export to ONNX
best = YOLO('bird_detector/feeder_v1/weights/best.pt')
best.export(format='onnx', imgsz=640, simplify=True)
print('\nExported to ONNX')

In [ ]:
# Verify ONNX output shape
import onnxruntime as ort
sess = ort.InferenceSession('bird_detector/feeder_v1/weights/best.onnx')
print('Input:', sess.get_inputs()[0].name, sess.get_inputs()[0].shape)
print('Output:', sess.get_outputs()[0].name, sess.get_outputs()[0].shape)
print('\nExpected: Input [1, 3, 640, 640], Output [1, 5, 8400]')

In [ ]:
# Download the trained model
files.download('bird_detector/feeder_v1/weights/best.onnx')